# Tracking Data + Git Workshop

**Triton Ball** · SkillCorner Open Data · kloppy

Today you will take one off-ball run from a real A-League match and turn it into an animated clip with speed, acceleration and distance to the nearest defender, all built from raw tracking data.

You will also ship every step as its own **branch → commit → push → pull request → merge**, so by the end Git is muscle memory.

| Part | What you build | Branch | PR |
|---|---|---|---|
| 1 | Load and explore tracking, events and phases | `feature/explore` | optional |
| 2 | Filter events down to off-ball runs (`src/events.py`) | `feature/off-ball-runs` | #2 |
| 3 | Speed, acceleration, direction, distances (`src/kinematics.py`) | `feature/kinematics` | #3 |
| 4 | Plot a single frame (`src/viz.py`) | `feature/frame-viz` | #4 |
| 5 | Animate the run (`src/viz.py`) | `feature/animation` | #5 |

**How this notebook works.** Grey code cells are worked examples: just run them. Cells marked **Your turn** have a `# YOUR CODE HERE`. Some tasks live in the `src/` Python files instead of the notebook; a **Check** cell after each one tells you if it works.

Data: [SkillCorner Open Data](https://github.com/SkillCorner/opendata). Please credit SkillCorner if you share anything you build.

## Part 0: Setup

Before running anything, make a branch for Part 1. In a terminal, from the repo folder:

```bash
git switch -c feature/explore
```

`git switch -c` creates a new branch and moves you onto it. Check with `git branch` (the `*` marks where you are).

In [ ]:
# autoreload: when you edit a file in src/, the notebook picks up the change
# without restarting the kernel.
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Make `src` importable whether you launched Jupyter from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from src import load, events, kinematics, viz

pd.set_option("display.max_columns", 40)

In [ ]:
# Where the data comes from. None = stream straight from SkillCorner's GitHub (needs internet).
# If the Wi-Fi is bad, a facilitator can hand you a local copy: set DATA_DIR to that folder.
DATA_DIR = os.environ.get("WORKSHOP_DATA_DIR")

load.list_matches(DATA_DIR)

In [ ]:
MATCH_ID = 1925299  # Brisbane Roar vs Perth Glory, 21 Dec 2024. Pick another id from the table if you like.

## Part 1: Load and explore

Three files per match:

| File | One row per | Key columns |
|---|---|---|
| `*_tracking_extrapolated.jsonl` | frame (10 per second) | x, y of every player and the ball |
| `*_dynamic_events.csv` | event | event_type, event_subtype, player, frame_start, frame_end |
| `*_phases_of_play.csv` | phase | team in possession, phase type, frame_start, frame_end |

`frame` is the glue between all three.

### 1a. Tracking data with kloppy

kloppy reads many providers' formats into one standard model. Loading a full match takes a minute or two.

In [ ]:
tracking = load.load_tracking(MATCH_ID, data_dir=DATA_DIR)
tracking

In [ ]:
meta = tracking.metadata
print("Teams:      ", [t.name for t in meta.teams])
print("Frame rate: ", meta.frame_rate, "fps")
print("Pitch:      ", meta.pitch_dimensions.pitch_length, "x", meta.pitch_dimensions.pitch_width, "m")
print("Periods:    ", [(p.id, p.start_timestamp, p.end_timestamp) for p in meta.periods])

players = load.player_lookup(tracking)
players.head(25)

In [ ]:
tdf = load.tracking_to_df(tracking)
print(tdf.shape)
tdf.head()

The DataFrame is **wide**: one row per frame, and every player gets two columns, `<player_id>_x` and `<player_id>_y`. A player who is not on the pitch has `NaN`.

Coordinates are in **meters** with the origin at the **center spot**: x runs along the length (−52.5 to 52.5 on a 105 m pitch), y along the width. Teams swap ends at halftime.

In [ ]:
# Your turn: how many frames are in each period, and how many minutes is that?
# Hint: tdf.groupby("period_id")["frame_id"].count(), and there are 10 frames per second.

# YOUR CODE HERE

### 1b. Dynamic events

SkillCorner derives these from the tracking data. There are about 300 columns, but only some are filled for each event type.

In [ ]:
ev = load.load_events(MATCH_ID, DATA_DIR)
print(ev.shape)
ev["event_type"].value_counts()

In [ ]:
# Which subtypes exist for each event type?
ev.groupby("event_type")["event_subtype"].value_counts()

In [ ]:
# Which columns are actually filled for each event type? (share of non-null values)
filled = ev.groupby("event_type").count().T / ev["event_type"].value_counts()
filled.loc[["frame_start", "player_name", "x_start", "distance_covered", "speed_avg",
            "targeted", "received", "dangerous", "pass_outcome", "xthreat",
            "separation_gain", "pressing_chain"]].round(2)

In [ ]:
# One off-ball run, all its filled columns
ev[ev["event_type"] == "off_ball_run"].iloc[0].dropna().head(60)

### 1c. Phases of play

Each row is a window of frames with the in-possession team's phase (build-up, create, finish, direct, quick break, transition, chaotic, set play) and the matching out-of-possession phase (high, medium, low block, ...). Phases only exist while the ball is in play.

In [ ]:
ph = load.load_phases(MATCH_ID, DATA_DIR)
ph[["index", "frame_start", "frame_end", "duration", "team_in_possession_shortname",
    "team_in_possession_phase_type", "team_out_of_possession_phase_type"]].head(10)

In [ ]:
# Your turn: for each in-possession phase type, how many phases are there
# and what is the average duration in seconds? Sort by count.

# YOUR CODE HERE

In [ ]:
# Your turn: which 5 players made the most off-ball runs in this match?

# YOUR CODE HERE

### Git checkpoint: `feature/explore`

Run these in a terminal from the repo folder (not in the notebook):

```bash
git status                         # what changed?
git diff notebooks/workshop.ipynb        # read your own changes before committing
git add notebooks/workshop.ipynb
git commit -m "Explore tracking, events and phases"
git push -u origin feature/explore
```

This first checkpoint is commit-only. Practice reading `git status` and `git log --oneline`. Then switch back to main for Part 2:

```bash
git switch main
```

> Tip: before committing a notebook, use *Kernel → Restart & Clear Output* (or install `nbstripout`) so the diff stays small.

## Part 2: From events to off-ball runs

```bash
git switch -c feature/off-ball-runs
```

SkillCorner classifies 10 run types: `dropping_off`, `coming_short`, `pulling_wide`, `pulling_half_space`, `support`, `run_ahead_of_the_ball`, `overlap`, `underlap`, `behind`, `cross_receiver`.

In [ ]:
obr = ev[ev["event_type"] == "off_ball_run"]
counts = obr["event_subtype"].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
counts.sort_values().plot.barh(ax=ax, color="#1f77b4")
ax.set_xlabel("runs")
ax.set_title(f"Off-ball runs by type, match {MATCH_ID}")
plt.show()

### Linking events to phases

Every event has a `phase_index` that points at the `index` column of the phases table. `events.add_phase_info` (already written, read it in `src/events.py`) merges the two so each run knows its phase's frame window.

In [ ]:
ev_ph = events.add_phase_info(ev, ph)
ev_ph.loc[ev_ph["event_type"] == "off_ball_run",
          ["event_id", "event_subtype", "frame_start", "phase_index",
           "team_in_possession_phase_type", "phase_frame_start", "phase_frame_end", "phase_duration"]].head()

In [ ]:
# Run types by phase: which runs happen when?
pd.crosstab(obr["event_subtype"], obr["team_in_possession_phase_type"])

### Your turn: `get_off_ball_runs` in `src/events.py`

Open `src/events.py` and implement `get_off_ball_runs`. The docstring tells you what each argument does. Save the file, then run the check cell below (autoreload picks up your change).

In [ ]:
# Check: run me after implementing get_off_ball_runs
r = events.get_off_ball_runs(ev)
assert (r["event_type"] == "off_ball_run").all(), "keep only off_ball_run events"
assert r["frame_start"].is_monotonic_increasing, "sort by frame_start"
assert list(r.index[:3]) == [0, 1, 2], "reset the index"

r = events.get_off_ball_runs(ev, subtype=["overlap", "underlap"], phase="create")
assert set(r["event_subtype"]) <= {"overlap", "underlap"}
assert set(r["team_in_possession_phase_type"]) <= {"create"}

r = events.get_off_ball_runs(ev, targeted=True, min_duration=2)
assert r["targeted"].all() and (r["duration"] >= 2).all()
print("All checks passed")

### Pick the run we will study

We want a run **in behind** that was **targeted** by a pass: the kind of run a coach wants to see on video. For match 1925299 we use event `1_181`: Trent Ostler (Perth Glory) runs about 50 m in 7 seconds on a quick break and receives the ball. For other matches we take the longest targeted run in behind.

In [ ]:
candidates = events.get_off_ball_runs(ev, subtype="behind", targeted=True)
candidates[events.RUN_COLUMNS].sort_values("distance_covered", ascending=False).head(10)

In [ ]:
RUN_ID = "1_181" if MATCH_ID == 1925299 else candidates.sort_values("distance_covered").iloc[-1]["event_id"]

run = ev.loc[ev["event_id"] == RUN_ID].iloc[0]
RUNNER = int(run["player_id"])
RUNNER_TEAM = int(run["team_id"])
F0, F1 = int(run["frame_start"]), int(run["frame_end"])

print(f"{run['player_name']} ({run['team_shortname']}): {run['event_subtype']} run, "
      f"{run['team_in_possession_phase_type']} phase")
print(f"frames {F0}-{F1} ({run['duration']} s), {run['distance_covered']} m, avg {run['speed_avg']} km/h")
print(f"from ({run['x_start']}, {run['y_start']}) to ({run['x_end']}, {run['y_end']})")

### Git checkpoint: `feature/off-ball-runs`

Run these in a terminal from the repo folder (not in the notebook):

```bash
git status                         # what changed?
git diff src/events.py        # read your own changes before committing
git add src/events.py notebooks/workshop.ipynb
git commit -m "Add get_off_ball_runs filter"
git push -u origin feature/off-ball-runs
```

Then on GitHub: open **your fork** → *Compare & pull request* → base `main` ← compare `feature/off-ball-runs` → fill in the template → **Create pull request**. Ask your neighbor to leave one review comment, then **Merge** and **Delete branch**.

Back in the terminal, start the next part from an up-to-date `main`:

```bash
git switch main
git pull
```

## Part 3: Tracking kinematics

```bash
git switch -c feature/kinematics
```

Tracking gives positions. Everything else is derived:

- **velocity** = change in position / change in time (m/s), split into `vx` and `vy`
- **speed** = length of the velocity vector, `sqrt(vx² + vy²)`
- **acceleration** = change in speed / change in time (m/s²)
- **direction** = angle of the velocity vector, `atan2(vy, vx)`

At 10 fps, one frame = 0.1 s. Positions jitter a little every frame, and differencing amplifies jitter, so we **smooth first**.

### 3a. Slice the tracking to the run

In [ ]:
PAD = 30  # 3 seconds of context either side
seq = tdf[(tdf["frame_id"] >= F0 - PAD) & (tdf["frame_id"] <= F1 + PAD)].reset_index(drop=True)

print(len(seq), "frames, periods:", seq["period_id"].unique())
gaps = seq["frame_id"].diff().dropna()
print("frame gaps larger than 1:", int((gaps > 1).sum()))

In [ ]:
# Worked example: one player's positions as a tidy table
xy = kinematics.player_xy(seq, RUNNER)
xy.head()

In [ ]:
# Raw speed WITHOUT smoothing, to see why we smooth
raw = xy.copy()
raw["speed_raw"] = np.hypot(raw["x"].diff(), raw["y"].diff()) * 10 * 3.6  # km/h
raw["speed_raw"].describe()

### Your turn: four functions in `src/kinematics.py`

Implement `smooth_positions`, `add_velocity`, `add_acceleration` and `add_direction`. Then run the check: it builds a fake player running in a straight line at exactly 5 m/s.

In [ ]:
# Check: run me after implementing the kinematics functions
fake = pd.DataFrame({"frame_id": range(20), "period_id": 1,
                     "x": np.arange(20) * 0.5,   # 0.5 m per frame = 5 m/s
                     "y": np.zeros(20)})
k = kinematics.add_direction(kinematics.add_acceleration(kinematics.add_velocity(
        kinematics.smooth_positions(fake, 3))))
mid = k.iloc[5:15]
assert np.allclose(mid["speed"], 5), "speed should be 5 m/s"
assert np.allclose(mid["speed_kmh"], 18), "5 m/s = 18 km/h"
assert np.allclose(mid["acceleration"], 0), "constant speed means zero acceleration"
assert np.allclose(mid["direction"], 0), "moving toward +x means direction 0"

gap = fake.drop(index=[6, 7])  # frames 6 and 7 missing
assert np.allclose(kinematics.add_velocity(gap)["speed"].iloc[6], 5), "use frame_id to get dt"
print("All checks passed")

In [ ]:
kin = kinematics.compute_kinematics(seq, RUNNER, window=5)
kin[["frame_id", "x", "y", "speed_kmh", "acceleration", "direction"]].iloc[PAD:PAD + 10].round(2)

In [ ]:
print(f"Our max speed: {kin['speed_kmh'].max():.1f} km/h")
print(f"Our avg speed during the run: {kin.loc[kin['frame_id'].between(F0, F1), 'speed_kmh'].mean():.1f} km/h")
print(f"SkillCorner's avg speed:      {run['speed_avg']:.1f} km/h")
# Sanity check: very few humans exceed ~36 km/h. A spike far above that is a tracking error.

### 3b. Distances between players

Distance is just Pythagoras on two positions. `kinematics.distances_from_player` does it for many players at once; first try one pair yourself.

In [ ]:
# Your turn: distance (m) between the runner and the player in possession, every frame of `seq`.
# The passer's id is run["player_in_possession_id"]. Columns are f"{id}_x" and f"{id}_y".

passer = int(run["player_in_possession_id"])
# YOUR CODE HERE
# dist_to_passer = ...

In [ ]:
opponents = players.loc[players["team_id"] != RUNNER_TEAM, "player_id"]
nearest = kinematics.nearest_opponent(seq, RUNNER, opponents)
nearest["ball_dist"] = kinematics.distance_to_ball(seq, RUNNER).to_numpy()
nearest.iloc[PAD:PAD + 5]

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.plot(kin["frame_id"], kin["speed_kmh"], color="#d62728", label="speed (km/h)")
ax1.set_xlabel("frame")
ax1.set_ylabel("speed (km/h)", color="#d62728")
ax2 = ax1.twinx()
ax2.plot(nearest["frame_id"], nearest["nearest_opponent_dist"], color="#1f77b4", label="nearest opponent (m)")
ax2.set_ylabel("distance to nearest opponent (m)", color="#1f77b4")
for f in (F0, F1):
    ax1.axvline(f, color="grey", ls="--", lw=1)
ax1.set_title(f"{run['player_name']}: speed vs separation (dashed = run start/end)")
plt.show()

### Git checkpoint: `feature/kinematics`

Run these in a terminal from the repo folder (not in the notebook):

```bash
git status                         # what changed?
git diff src/kinematics.py        # read your own changes before committing
git add src/kinematics.py notebooks/workshop.ipynb
git commit -m "Compute speed, acceleration and direction"
git push -u origin feature/kinematics
```

Then on GitHub: open **your fork** → *Compare & pull request* → base `main` ← compare `feature/kinematics` → fill in the template → **Create pull request**. Ask your neighbor to leave one review comment, then **Merge** and **Delete branch**.

Back in the terminal, start the next part from an up-to-date `main`:

```bash
git switch main
git pull
```

## Part 4: Visualize a frame

```bash
git switch -c feature/frame-viz
```

`viz.draw_pitch` draws an empty pitch in SkillCorner coordinates with mplsoccer. Your job is to put the players on it.

In [ ]:
pitch_len = tracking.metadata.pitch_dimensions.pitch_length
pitch_wid = tracking.metadata.pitch_dimensions.pitch_width
pitch, fig, ax = viz.draw_pitch(pitch_len, pitch_wid)
plt.show()

### Your turn: `plot_frame` in `src/viz.py`

Fill in the three TODO blocks: home and away players, the highlighted runner (plus a velocity arrow as a bonus), and the ball. Then draw the run's first and last frames side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
viz.plot_frame(tdf, F0, players, highlight=RUNNER, kinematics=kin,
               pitch_length=pitch_len, pitch_width=pitch_wid, ax=axes[0], title=f"Run starts (frame {F0})")
viz.plot_frame(tdf, F1, players, highlight=RUNNER, kinematics=kin,
               pitch_length=pitch_len, pitch_width=pitch_wid, ax=axes[1], title=f"Run ends (frame {F1})")
plt.show()

# Check: the plot should have at least 3 groups of points (home, away, runner)
assert len(axes[0].collections) >= 3, "did you scatter home, away and the runner?"

### Git checkpoint: `feature/frame-viz`

Run these in a terminal from the repo folder (not in the notebook):

```bash
git status                         # what changed?
git diff src/viz.py        # read your own changes before committing
git add src/viz.py notebooks/workshop.ipynb
git commit -m "Plot a tracking frame"
git push -u origin feature/frame-viz
```

Then on GitHub: open **your fork** → *Compare & pull request* → base `main` ← compare `feature/frame-viz` → fill in the template → **Create pull request**. Ask your neighbor to leave one review comment, then **Merge** and **Delete branch**.

Back in the terminal, start the next part from an up-to-date `main`:

```bash
git switch main
git pull
```

## Part 5: Animate the sequence

```bash
git switch -c feature/animation
```

`FuncAnimation` calls an `update(i)` function once per frame. Your job is to fill in `update` inside `viz.animate_sequence` so the players, ball, runner and trail move.

In [ ]:
anim = viz.animate_sequence(tdf, F0 - PAD, F1 + PAD, players, highlight=RUNNER, kinematics=kin,
                            trail=15, pitch_length=pitch_len, pitch_width=pitch_wid)
HTML(anim.to_jshtml())

In [ ]:
# Save it as a GIF (GIFs are in .gitignore, so you won't commit them by accident)
from matplotlib import animation
anim.save(ROOT / "run.gif", writer=animation.PillowWriter(fps=10))
print("saved", ROOT / "run.gif")

### Stretch goals (pick one)

1. Animate the **5 fastest** runs of the match (sort `get_off_ball_runs(ev)` by `speed_avg`).
2. Color every player by their current speed instead of team.
3. Draw a line from the player in possession to the runner, so you can see the passing lane open.
4. Compare your speed curve with SkillCorner's `speed_avg` for 20 runs. How close are you? What smoothing window gets closest?

In [ ]:
# Stretch goal workspace

# YOUR CODE HERE

### Git checkpoint: `feature/animation`

Run these in a terminal from the repo folder (not in the notebook):

```bash
git status                         # what changed?
git diff src/viz.py        # read your own changes before committing
git add src/viz.py notebooks/workshop.ipynb
git commit -m "Animate an off-ball run"
git push -u origin feature/animation
```

Then on GitHub: open **your fork** → *Compare & pull request* → base `main` ← compare `feature/animation` → fill in the template → **Create pull request**. Ask your neighbor to leave one review comment, then **Merge** and **Delete branch**.

Back in the terminal, start the next part from an up-to-date `main`:

```bash
git switch main
git pull
```

Paste your GIF into the PR description (drag and drop it into the text box on GitHub). That's your portfolio piece.

## Wrap-up

You loaded broadcast tracking with kloppy, filtered SkillCorner's dynamic events to the runs you care about, turned positions into speed, acceleration, direction and distances, and animated it, shipping each step through a pull request.

**Next steps:** try another match, write a `pressure_on_runner` metric (number of opponents within 5 m), or join this with the season aggregates in the SkillCorner repo's `data/aggregates` folder.